In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.triathlon_utils import (
    parse_fit, 
    clean_fit_data, 
    calculate_elapsed_time, 
    apply_triathlon_style,
    format_time_axis,
    export_for_instagram,
    KLEUREN_PALETTE
)

from utils.config import DEFAULT_ATHLETE, DEFAULT_PATHS

In [0]:
def calculate_vam_metrics(df: pd.DataFrame, rolling_window_sec: int = 120) -> pd.DataFrame:
    """
    Berekent de VAM (Velocità Ascensionale Media in m/u) over een rollend tijdvenster.
    
    Formule: VAM = (stijging in meters / tijd in seconden) * 3600
    """
    df = df.copy()

    # Zorg dat hoogte correct aanwezig is
    if 'enhanced_altitude' in df.columns:
        df['altitude'] = df['enhanced_altitude']
    elif 'altitude' not in df.columns:
        df['altitude'] = 0.0

    # Tijd- en hoogteverschillen berekenen
    df['time_diff'] = df['timestamp'].diff().dt.total_seconds()
    df['alt_diff'] = df['altitude'].diff()

    # Alleen positieve hoogtemeters meenemen (stijging)
    df['gain'] = np.where(df['alt_diff'] > 0, df['alt_diff'], 0)

    # Rolling window berekening op basis van tijd
    df_time = df.set_index('timestamp')
    rolling_gain = df_time['gain'].rolling(f'{rolling_window_sec}s').sum()
    rolling_time = df_time['time_diff'].rolling(f'{rolling_window_sec}s').sum()

    df['vam'] = np.where(rolling_time > 0, (rolling_gain / rolling_time) * 3600, 0)
    df['vam'] = df['vam'].clip(lower=0, upper=3000)
    df['total_ascent'] = df['gain'].cumsum()

    # Afstand in km toevoegen indien aanwezig
    if 'distance' in df.columns and df['distance'].notna().any():
        df['distance_km'] = df['distance'] / 1000.0

    return df

In [0]:
def plot_vam_dashboard(df, window_size=300):
    apply_triathlon_style()
    
    df_clean = df.copy()
        
    # 1. Herbereken actieve tijd
    if 'time_diff' in df_clean.columns:
        active_time_diff = df_clean['time_diff'].clip(upper=5) 
        df_clean['active_elapsed_minutes'] = active_time_diff.cumsum() / 60.0
        x_col = 'active_elapsed_minutes'
    elif 'elapsed_minutes' in df_clean.columns:
        x_col = 'elapsed_minutes'
    else:
        df_clean['step'] = np.arange(len(df_clean))
        x_col = 'step'

    df_clean = df_clean.reset_index(drop=True)
    
    # 2. Bereken de hellingshoek (gradient %) als deze nog niet bestaat
    if 'gradient' not in df_clean.columns and 'distance' in df_clean.columns:
        dist_diff = df_clean['distance'].diff().replace(0, np.nan)
        alt_diff = df_clean['altitude'].diff()
        df_clean['gradient'] = (alt_diff / dist_diff) * 100

    # 3. Extra smoothing van hoogte om ruis-pieken in VAM te voorkomen
    alt_smoothed = df_clean['altitude'].rolling(window=15, min_periods=1, center=True).mean()
    
    # 4. Smooth VAM & filter stilstand + afdalingen
    vam_raw = df_clean['vam'].copy()
    
    # Zet VAM op NaN als je daalt of bijna stilstaat (< 1% stijging)
    if 'gradient' in df_clean.columns:
        vam_raw = np.where(df_clean['gradient'] >= 1.0, vam_raw, np.nan)
    
    # Maak er een Series van om rolling mean toe te passen met behoud van NaNs
    vam_series = pd.Series(vam_raw)
    df_clean['vam_smooth'] = vam_series.rolling(window=window_size, min_periods=30).mean()

    hr_col = 'heart_rate' if 'heart_rate' in df_clean.columns else 'hr'
    if hr_col in df_clean.columns:
        df_clean['hr_smooth'] = df_clean[hr_col].rolling(window=60, min_periods=1).mean()

    # --- GRAFIEK MAKEN ---
    fig, ax_vam = plt.subplots(figsize=(13.33, 7.5), dpi=300)

    COLOR_ELEVATION = '#8A99AD'
    COLOR_VAM       = '#00F5FF'
    COLOR_HR        = '#FF2A6D'

    # 1. HOOGTEPROFIEK (Zachte achtergrond)
    ax_alt = ax_vam.twinx()
    ax_alt.fill_between(
        df_clean[x_col], df_clean['altitude'], df_clean['altitude'].min(), 
        color=COLOR_ELEVATION, alpha=0.15, zorder=1
    )
    ax_alt.plot(df_clean[x_col], df_clean['altitude'], color=COLOR_ELEVATION, linewidth=1.0, alpha=0.4, zorder=1)

    ax_alt.set_ylabel('')              
    ax_alt.set_yticks([])              
    ax_alt.yaxis.set_visible(False)

    # 2. VAM (Primaire linker Y-as)
    ax_vam.plot(df_clean[x_col], df_clean['vam_smooth'], color=COLOR_VAM, linewidth=2.2, label=f'VAM ({window_size}s gem.)', zorder=3)
    ax_vam.set_ylabel('VAM (m/u)', color=COLOR_VAM, fontweight='bold')
    ax_vam.tick_params(axis='y', labelcolor=COLOR_VAM)
    ax_vam.set_zorder(ax_alt.get_zorder() + 1)
    ax_vam.patch.set_visible(False)

    valid_vam = df_clean['vam_smooth'].dropna()
    max_vam = int(valid_vam.max()) if not valid_vam.empty else 0
    avg_vam = int(valid_vam.mean()) if not valid_vam.empty else 0
    if not valid_vam.empty:
        ax_vam.set_ylim(0, max_vam * 1.25)

    avg_hr_text = "N/A"
    if hr_col in df_clean.columns and df_clean[hr_col].notna().any():
        avg_hr = df_clean[hr_col].mean()
        avg_hr_text = f"{int(avg_hr)} BPM"

    # 3. HARTSLAG (Secundaire rechter Y-as)
    if hr_col in df_clean.columns and df_clean[hr_col].notna().any():
        ax_hr = ax_vam.twinx()
        ax_hr.spines['right'].set_position(('outward', 10))
        
        hr_data = df_clean['hr_smooth']
        hr_min = int(hr_data.min() - 5)
        hr_max = int(hr_data.max() + 5)

        ax_hr.plot(df_clean[x_col], hr_data, color=COLOR_HR, linewidth=1.8, label='Hartslag (BPM)', zorder=4)
        ax_hr.set_ylabel('Hartslag (BPM)', color=COLOR_HR, fontweight='bold')
        ax_hr.tick_params(axis='y', labelcolor=COLOR_HR)
        ax_hr.set_ylim(hr_min - 10, hr_max + 10)

    # Statistieken overlay box
    total_elevation = int(df_clean['altitude'].max() - df_clean['altitude'].min())
    stats_text = (
        f"Totale stijging: +{total_elevation}m\n"
        f"Avg VAM: {avg_vam} m/h\n"
        f"Max VAM: {max_vam} m/h\n"
        f"Avg Hartslag: {avg_hr_text}"
    )

    ax_vam.text(
        0.02, 0.95, stats_text, 
        transform=ax_vam.transAxes, 
        fontsize=10, color='white', fontweight='bold',
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#111827', alpha=0.85, edgecolor='#374151')
    )

    # X-as & Titel
    ax_vam.set_xlabel('Actieve Tijd (minuten)', fontweight='bold', color=KLEUREN_PALETTE.get('text', '#FFF'))
    fig.suptitle('ETAPPE PERFORMANCE: KLIM-INTENSITEIT (VAM) vs. INSPANNING (Hartslag)', fontsize=14, fontweight='bold', y=0.96, color=KLEUREN_PALETTE.get('text', '#FFF'))
    
    plt.tight_layout()
    return fig

In [0]:
fit_file_path = f"{DEFAULT_PATHS.fit_dir}/20260912_stubaier_hohenweg_etappe_1.fit"

# 1. Inlezen & Opschonen
df = parse_fit(fit_file_path)
df_clean = clean_fit_data(df)
df_active = calculate_elapsed_time(df_clean)

# 3. VAM berekenen en plotten
df_vam = calculate_vam_metrics(df_active, rolling_window_sec=300)

# 1. Genereer dashboard
fig = plot_vam_dashboard(df_vam, window_size=300)

# 2. Exporteer naar landscape formaat 
export_for_instagram(fig, filename="20260912_vam_etappe_1", format_type='landscape')